#### silver-layer dataset;

- Assuming all datasets were properly masked for PII columns and for specific rows based on conditions like geo-locations,
next step is to create the silver-layer;

- silver-layer consists of datasets pre-AI models; or pre-Gold layer ready for visualization;


https://docs.databricks.com/aws/en/optimizations/predictive-optimization

##### Applying mask

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def mask_value(value):
    if value is None:
        return None
    # Mask email addresses
    if '@' in value:
        local, domain = value.split('@', 1)
        masked_local = '*' * (len(local) - 2) if len(local) > 2 else '*' * len(local)
        masked_domain = '*' * (len(domain) - 4) + domain[-4:] if len(domain) > 4 else '*' * len(domain)
        return f"{masked_local}@{masked_domain}"
    # Mask names (first/last)
    if len(value) > 1:
        return '*' * (len(value) - 1)
    return '*' * len(value)

mask_value_udf = udf(mask_value, StringType())
spark.udf.register("mask_value", mask_value, StringType())

In [0]:
%sql
SELECT * FROM end_to_end_retail.db_landing.tbl_clicked_items limit 5

In [0]:
%sql
SELECT * FROM end_to_end_retail.db_landing.tbl_customer_autoloader limit 5

- SET spark.databricks.sql.dsv2.unique.enabled = true;


In [0]:
%sql
    
CREATE OR REPLACE TABLE end_to_end_retail.db_silver.tbs_customer_address_clicks_geoloc (
  customer_id STRING PRIMARY KEY,
  ship_to_address STRING,
  state STRING,
  city STRING,
  region STRING,
  district STRING,
  clicked_items STRING,
  lon DOUBLE,
  lat DOUBLE,
  valid_from STRING,
  valid_to STRING,
  insert_date DATE
  -- CONSTRAINT customer_id_unique UNIQUE (customer_id)
);

INSERT INTO end_to_end_retail.db_silver.tbs_customer_address_clicks_geoloc
SELECT
  a.customer_id,
  g.ship_to_address,
  a.state,
  a.city,
  a.region,
  a.district,
  b.clicked_items,
  ROUND(CAST(g.lon AS DOUBLE), 2) AS lon,
  ROUND(CAST(g.lat AS DOUBLE), 2) AS lat,
  g.valid_from,
  g.valid_to,
  current_timestamp() as insert_date_orders_changes
FROM end_to_end_retail.db_landing.tbl_customer_address a
LEFT JOIN end_to_end_retail.db_landing.tbl_clicked_items b
  ON a.customer_id = b.customer_id
LEFT JOIN end_to_end_retail.db_landing.tbl_customer_geoloc g
  ON a.customer_id = g.customer_id
-- WHERE b.customer_id IS NOT NULL

- Here a good dataset for clustering techniques, lat/lon, distance; specially clicks per region, worth finding insights 

- customer ID does not match:
- SELECT * FROM end_to_end_retail.db_landing.tbl_customer_autoloader